In [1]:
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    fetch_and_parse,
    coerce_types,
)

/Users/alirachidi/miniconda3/lib/python3.9/site-packages/girder_client/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    rows = []
    limit = 50
    offset = 0
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": "pdv_alpss_results",
                },
            )
            if not batch:
                break
            futures = [executor.submit(fetch_and_parse, gc, item) for item in batch]
            for future in as_completed(futures):
                rows.append(future.result())
            if len(batch) < limit:
                break
            offset += limit

    df = pd.DataFrame(rows)
    df = coerce_types(df)

/Users/alirachidi/dev/work/aimdl/aimdl_examples/src/aimdl_examples/download.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [ ]:
df.head()


,Date,File Name,Velocity OK,Spall OK,Uncertainty OK,Error Message,Run Time,Velocity at Max Compression,Time at Max Compression,Velocity at Max Tension,...,HEL Detected,HEL Strength (GPa),HEL Uncertainty (GPa),HEL Free Surface Velocity (m/s),HEL Time Detection (ns),HEL Consecutive Points,HEL Segment Duration (ns),HEL Strain Rate,igsn,itemId
0,2026-03-17 21:18:00,C1--20251022--00002.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.864054,760.198712,0.000001,-146.270060,...,False,NaN,NaN,-7.376598,968.859375,12,0.085938,NaN,JHAMAB00021-16,69b9c5134d9c815b961aa2ec
1,2026-03-17 21:17:00,C1--20251022--00009.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.848154,1092.321038,0.000001,-123.587673,...,False,NaN,NaN,-3.337541,980.039062,101,0.781250,NaN,JHAMAB00021-16,69b9c4ea4e69e623f00204be
2,2026-03-17 21:17:00,C1--20251022--00010.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.846242,1094.988503,0.000001,124.939287,...,False,NaN,NaN,-8.952012,1063.843750,10,0.070313,NaN,JHAMAB00021-16,69b9c4e54d9c815b961aa2cd
3,2026-03-17 21:17:00,C1--20251022--00006.csv,True,False,False,spall: index 23 is out of bounds for axis 0 wi...,0 days 00:00:00.845721,NaN,NaN,NaN,...,False,NaN,NaN,1.284756,979.593750,187,1.453125,NaN,JHAMAB00021-16,69b9c4fc4e69e623f00204fc
4,2026-03-17 21:17:00,C1--20251022--00005.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.842115,1002.092647,0.000001,-81.444346,...,False,NaN,NaN,-5.029551,1031.539062,14,0.101563,NaN,JHAMAB00021-16,69b9c5027590bb7e1ed223c9


In [8]:
df_filtered = df[df["File Name"] == "C1--20251022--00002.csv"]
df_filtered.head()

,Date,File Name,Velocity OK,Spall OK,Uncertainty OK,Error Message,Run Time,Velocity at Max Compression,Time at Max Compression,Velocity at Max Tension,...,HEL Detected,HEL Strength (GPa),HEL Uncertainty (GPa),HEL Free Surface Velocity (m/s),HEL Time Detection (ns),HEL Consecutive Points,HEL Segment Duration (ns),HEL Strain Rate,igsn,itemId
0,2026-03-17 21:18:00,C1--20251022--00002.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.864054,760.198712,0.000001,-146.27006,...,False,NaN,NaN,-7.376598,968.859375,12,0.085938,NaN,JHAMAB00021-16,69b9c5134d9c815b961aa2ec
